# Código de traductores automáticos: GoogleTranslator y MyMemoryTranslator

En este cuaderno vamos a traducir automáticamente un texto original (TO) del inglés al español con la librería `deep-translator`, que permite usar varios motores de traducción automática desde Python. Veremos dos opciones:

1. **Opción A: `GoogleTranslator`.** Es la que usamos en clase. Traduce el texto completo de una sola vez.
2. **Opción B: `MyMemoryTranslator`.** Es la alternativa cuando Google da el error `TooManyRequests`. Como MyMemory solo admite **500 caracteres por petición**, antes hay que **dividir el TO en bloques**, sin cortar frases por la mitad.

Al final, sea cual sea el motor, la traducción queda guardada en la misma variable (`TMA1`), así que el código que venga después (por ejemplo, el de las métricas de evaluación) funciona igual con las dos opciones.

## 0. El texto original (TO1)

In [ ]:
TO1 = """A brief history of insulin
Diabetes has been known since antiquity: an Egyptian papyrus from around 1550 BC already describes a disease marked by excessive urination, and the Greek physician Aretaeus of Cappadocia later gave it the name "diabetes". For centuries, however, physicians could do little more than describe its symptoms and recommend strict diets, and a diagnosis of type 1 diabetes usually meant death within a few months or years.
In 1921, Frederick Banting and Charles Best, working in John Macleod's laboratory at the University of Toronto, managed to extract an active substance from the pancreas of dogs. With the help of the biochemist James Collip, the extract was purified, and in January 1922 a 14-year-old boy named Leonard Thompson became the first person to receive an insulin injection. His blood glucose fell dramatically, and within months insulin was being produced on an industrial scale. In 1923, Banting and Macleod were awarded the Nobel Prize in Physiology or Medicine.
"""

Guardamos el texto original en la variable `TO1`. Las **triples comillas** (`"""…"""`) permiten escribir un texto de varias líneas y conservan los saltos de línea. En este texto, cada línea es un párrafo: el título y tres párrafos de contenido.

Antes de traducir, conviene hacer una **radiografía del texto**: cuántos caracteres tiene en total y cuántos tiene cada párrafo. Lo necesitaremos para entender por qué MyMemory obliga a trocearlo.

In [ ]:
# Longitud total del TO (en caracteres, contando espacios y signos)
print("Caracteres del TO1:", len(TO1))

# Longitud de cada párrafo: dividimos por saltos de línea (\n) y descartamos las líneas vacías
for i, parrafo in enumerate(TO1.split("\n"), 1):
    if parrafo.strip():
        print(f"Párrafo {i}: {len(parrafo.strip())} caracteres")

**Qué vemos:** el TO tiene 1004 caracteres en total, repartidos en tres líneas: el título (26 caracteres) y dos párrafos (417 y 558 caracteres).

- Para **Google** no hay problema: admite textos mucho más largos en una sola petición.
- Para **MyMemory** sí lo hay: el texto completo supera con creces los 500 caracteres. Si se lo enviamos entero, devuelve el error `NotValidLength`. Y no basta con enviarlo párrafo a párrafo, porque **el segundo párrafo (558 caracteres) tampoco cabe**: habrá que partirlo.

## 1. Opción A: traducción con `GoogleTranslator`

In [ ]:
# Instalamos la librería (en Google Colab, cambia % por !)
%pip install deep-translator

In [ ]:
# Usaremos GoogleTranslator para traducir el texto del inglés al español
from deep_translator import GoogleTranslator  # Importamos la clase que realizará la TA

In [ ]:
TMA1 = GoogleTranslator(source='en', target='es').translate(TO1)  # Llamamos a la API para traducir el texto
print("Traducción automática:", TMA1)  # Mostramos en pantalla el resultado de la traducción automática

**Qué hace este código:**

- `GoogleTranslator(source='en', target='es')` crea un traductor del inglés (`en`) al español (`es`).
- `.translate(TO1)` envía **todo el texto de una vez** al servicio de Google y devuelve la traducción, que guardamos en `TMA1` (traducción mediante máquina del texto 1).
- `print()` muestra el resultado.

⚠️ **Si aparece el error `TooManyRequests`**: el código está bien. Lo que ocurre es que el servidor de Google ha recibido demasiadas peticiones desde la misma red (por ejemplo, la de la universidad, cuando toda la clase traduce a la vez) y la bloquea temporalmente. En ese caso, pasa a la **opción B**.

## 2. Opción B: traducción con `MyMemoryTranslator` (si Google da `TooManyRequests`)

MyMemory es otro servicio de traducción automática gratuito al que también se accede con `deep-translator`. Tiene una limitación importante: **cada petición admite como máximo 500 caracteres**. Por eso no podemos enviarle el TO entero, sino trozo a trozo.

¿Cómo trocear sin estropear la traducción? Siguiendo dos reglas:

1. **No cortar nunca una frase por la mitad.** Un motor de traducción necesita la frase completa para entender su estructura. Si cortáramos a los 500 caracteres exactos, partiríamos frases y hasta palabras, y la traducción saldría incoherente.
2. **Respetar los párrafos del original**, para que la traducción conserve la misma estructura que el TO.

Lo hacemos en tres pasos: definir una función que trocea, aplicarla al TO y traducir bloque a bloque.

### Paso 1. Definir la función `trocear()`

La función recibe un texto y devuelve una **lista de bloques**, cada uno de menos de `max_chars` caracteres (por defecto, 450, para dejar margen respecto a los 500 de MyMemory).

Funciona así:

1. **Divide el texto en frases** con `re.split(r'(?<=[.!?])\s+', …)`. La expresión regular busca los espacios o saltos de línea (`\s+`) que van **justo detrás** de un punto, una exclamación o una interrogación. La parte `(?<=[.!?])` solo «mira hacia atrás»: comprueba que antes hay un signo de final de frase, pero no lo elimina. Así cada frase conserva su punto.
2. **Va llenando un bloque** (`actual`) frase a frase, mientras quepa.
3. Cuando la siguiente frase **ya no cabe**, guarda el bloque en la lista `bloques` y empieza uno nuevo con esa frase.
4. Al terminar, **guarda el último bloque**, que si no se perdería.

In [ ]:
import re

def trocear(texto, max_chars=450):
    """Divide un texto en bloques de menos de max_chars caracteres sin cortar frases."""
    # 1. Separar el texto en frases: cortamos tras . ! ? seguidos de espacio o salto de línea
    frases = re.split(r'(?<=[.!?])\s+', texto.strip())

    bloques, actual = [], ""
    for frase in frases:
        # 2. Si la frase cabe en el bloque actual, se añade
        if len(actual) + len(frase) + 1 <= max_chars:
            actual = f"{actual} {frase}".strip()
        # 3. Si no cabe, se cierra el bloque actual y se empieza uno nuevo
        else:
            if actual:
                bloques.append(actual)
            actual = frase
    # 4. No olvidar el último bloque
    if actual:
        bloques.append(actual)
    return bloques

Para ver cómo funciona, la probamos con un ejemplo pequeño y un límite muy bajo (30 caracteres). Observa que ninguna frase se corta: cuando una frase no cabe en el bloque actual, pasa entera al siguiente.

In [ ]:
ejemplo = "Frase uno. La frase dos es más larga. Frase tres."
for bloque in trocear(ejemplo, max_chars=30):
    print(f"[{len(bloque)} caracteres] {bloque}")

### Paso 2. Trocear el TO1 párrafo a párrafo

En lugar de trocear el texto entero, recorremos **cada párrafo por separado** (`TO1.split("\n")`) y troceamos cada uno. Así un bloque nunca mezcla frases de dos párrafos distintos, y después podremos reconstruir la estructura del original.

`extend()` añade a la lista `TO1_BLOQUES` todos los bloques que devuelve `trocear()` para ese párrafo. Al final comprobamos que ningún bloque supera los 500 caracteres.

In [ ]:
# Troceamos por párrafos para respetar la estructura del original
TO1_BLOQUES = []
for parrafo in TO1.split("\n"):
    if parrafo.strip():
        TO1_BLOQUES.extend(trocear(parrafo))

# Comprobación: ningún bloque debe superar 500 caracteres
for i, b in enumerate(TO1_BLOQUES, 1):
    print(f"Bloque {i}: {len(b)} caracteres")

In [ ]:
# Veamos qué contiene cada bloque
for i, b in enumerate(TO1_BLOQUES, 1):
    print(f"--- Bloque {i} ({len(b)} caracteres) ---")
    print(b, "\n")

In [ ]:
print(TO1_BLOQUES)

**Qué vemos:** salen **cuatro bloques a partir de tres líneas**. El título (26 caracteres) y el primer párrafo (417) caben enteros en un bloque cada uno. El segundo párrafo, de 558 caracteres, supera el límite, así que `trocear()` lo ha dividido en **dos bloques (367 + 190)**. Ha cortado justo después de «…the first person to receive an insulin injection.», es decir, al final de una frase, sin partir ninguna por la mitad.

El resultado es exactamente la misma lista que podríamos haber escrito a mano. Lo comprobamos:

In [ ]:
TO1_BLOQUES_MANUAL = [
    "A brief history of insulin",
    'Diabetes has been known since antiquity: an Egyptian papyrus from around 1550 BC already describes a disease marked by excessive urination, and the Greek physician Aretaeus of Cappadocia later gave it the name "diabetes". For centuries, however, physicians could do little more than describe its symptoms and recommend strict diets, and a diagnosis of type 1 diabetes usually meant death within a few months or years.',
    "In 1921, Frederick Banting and Charles Best, working in John Macleod's laboratory at the University of Toronto, managed to extract an active substance from the pancreas of dogs. With the help of the biochemist James Collip, the extract was purified, and in January 1922 a 14-year-old boy named Leonard Thompson became the first person to receive an insulin injection.",
    "His blood glucose fell dramatically, and within months insulin was being produced on an industrial scale. In 1923, Banting and Macleod were awarded the Nobel Prize in Physiology or Medicine.",
]

print("¿Coinciden los bloques automáticos con la lista manual?", TO1_BLOQUES == TO1_BLOQUES_MANUAL)

La ventaja de la función frente a la lista manual es que **sirve para cualquier texto**: si mañana el TO es otro, no hay que volver a cortarlo a mano.

### Paso 3. Traducir cada bloque con MyMemory y reconstruir el texto

- `MyMemoryTranslator(source='en-US', target='es-ES')` crea el traductor. Fíjate en que MyMemory usa códigos de idioma con variante regional (`en-US`, `es-ES`), a diferencia de Google (`en`, `es`).
- `traductor.translate(b) for b in TO1_BLOQUES` traduce los bloques **uno a uno**: es una petición por bloque, y cada una cumple el límite de 500 caracteres.
- `"\n".join(…)` vuelve a unir las traducciones, separándolas con un salto de línea. Ojo: como el segundo párrafo se dividió en dos bloques, en la traducción aparecerá **partido en dos líneas**. El contenido es el mismo; si se quiere recuperar exactamente la estructura del original, basta con unir esas dos líneas en un solo párrafo.

In [ ]:
from deep_translator import MyMemoryTranslator

traductor = MyMemoryTranslator(source='en-US', target='es-ES')
TMA1 = "\n".join(traductor.translate(b) for b in TO1_BLOQUES)
print("Traducción automática:\n", TMA1)

### Paso 4. Comprobar la correspondencia entre original y traducción

Para revisar la traducción como lo haría un traductor profesional, mostramos cada bloque del original junto a su traducción.

In [ ]:
for i, (original, traduccion) in enumerate(zip(TO1_BLOQUES, TMA1.split("\n")), 1):
    print(f"--- Bloque {i} ---")
    print("TO :", original)
    print("TMA:", traduccion, "\n")

## Resumen

| paso | qué hacemos | por qué |
|---|---|---|
| 0 | Guardar el TO y medir su longitud | Saber si cabe en una sola petición |
| 1A | Traducir con `GoogleTranslator` | Traduce el texto completo de una vez |
| 1B | Si Google da `TooManyRequests`, pasar a `MyMemoryTranslator` | Es otro motor que sí responde, pero con un límite de 500 caracteres por petición |
| 2 | Trocear el TO por párrafos y por frases (`trocear()`) | Cumplir el límite sin cortar frases ni mezclar párrafos |
| 3 | Traducir bloque a bloque y unir con `"\n"` | Obtener la traducción completa con la estructura del original |
| 4 | Comparar bloque a bloque | Revisar la traducción |

📌 **Ten en cuenta:** Google y MyMemory son motores distintos, así que **no producen la misma traducción**. Si después evalúas la traducción con métricas (precisión, recall, BLEU, chrF…), los valores cambiarán según el motor que hayas usado. Lo importante es interpretar los resultados que obtengas tú.